## 가설 설정

### 1. 성별(Gender)에 따른 가설
- H0: 성별에 따라 bogo/discount 전환율 차이가 없다
- H1: 성별에 따라 bogo/discount 전환율 차이가 있다

### 2. 연령대(Age)에 따른 가설
- H0: 연령대에 따라 bogo/discount 전환율 차이가 없다
- H1: 연령대에 따라 bogo/discount 전환율 차이가 있다

### 3. 소득수준(Income)에 따른 가설
- H0: 소득수준에 따라 bogo/discount 전환율 차이가 없다
- H1: 소득수준에 따라 bogo/discount 전환율 차이가 있다

### 4. 같은 고객층 안에서 bogo와 discount 차이
- H0: 같은 고객층 안에서 bogo와 discount 전환율 차이가 없다
- H1: 같은 고객층 안에서 bogo와 discount 전환율 차이가 있다


In [34]:
# 필요한 라이브러리 불러오기
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display


In [35]:
# 데이터 불러오기
promotion_final = pd.read_csv('./data/promotion_final.csv')

# 확인
promotion_final.head()


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,is_deduplicated,completion_amount_current,promo_influenced_amount,prev_received,completion_amount_prev_received,prev_viewed,promo_influenced_amount_prev_received,adjusted_completion_amount,amount_raw,promo_influenced_amount_final
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,0,8.57,0.0,NaN,NaN,NaN,NaN,8.57,8.57,0.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,0,14.11,0.0,NaN,NaN,NaN,NaN,14.11,14.11,0.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,0,10.27,NaN,NaN,NaN,NaN,NaN,10.27,10.27,NaN
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,informational,0,0,3,...,0,0.00,0.0,NaN,NaN,NaN,NaN,0.00,NaN,0.0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,informational,0,0,4,...,0,0.00,0.0,NaN,NaN,NaN,NaN,0.00,NaN,0.0


In [36]:
# 컬럼 확인
promotion_final.columns


Index(['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed',
       'offer completed', 'offer_type', 'difficulty', 'reward', 'duration',
       'web', 'email', 'mobile', 'social', 'gender', 'age', 'became_member_on',
       'income', 'amount', 'is_received', 'is_viewed', 'is_completed',
       'is_normal_flow', 'is_deduplicated', 'completion_amount_current',
       'promo_influenced_amount', 'prev_received',
       'completion_amount_prev_received', 'prev_viewed',
       'promo_influenced_amount_prev_received', 'adjusted_completion_amount',
       'amount_raw', 'promo_influenced_amount_final'],
      dtype='str')

In [37]:
# bogo와 discount만 남기기
bogo_dis = promotion_final[promotion_final['offer_type'].isin(['bogo', 'discount'])].copy()

# 확인
print(bogo_dis.shape)
display(bogo_dis.head())


(61042, 33)


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,is_deduplicated,completion_amount_current,promo_influenced_amount,prev_received,completion_amount_prev_received,prev_viewed,promo_influenced_amount_prev_received,adjusted_completion_amount,amount_raw,promo_influenced_amount_final
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,0,8.57,0.0,NaN,NaN,NaN,NaN,8.57,8.57,0.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,0,14.11,0.0,NaN,NaN,NaN,NaN,14.11,14.11,0.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,0,10.27,NaN,NaN,NaN,NaN,NaN,10.27,10.27,NaN
5,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,bogo,5,5,5,...,0,0.00,0.0,NaN,NaN,NaN,NaN,0.00,NaN,NaN
6,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,bogo,5,5,5,...,0,0.00,0.0,NaN,NaN,NaN,NaN,0.00,NaN,NaN


In [38]:
# is_deduplicated를 최종 전환 타겟으로 사용
bogo_dis['is_deduplicated'] = bogo_dis['is_deduplicated'].fillna(0).astype(int)

# 확인
bogo_dis['is_deduplicated'].value_counts()


is_deduplicated
0    38709
1    22333
Name: count, dtype: int64

In [39]:
# age_group 만들기
bogo_dis['age_group'] = pd.cut(
    bogo_dis['age'],
    bins=[0, 19, 29, 39, 49, 59, 69, 120],
    labels=['10s', '20s', '30s', '40s', '50s', '60s', '70+']
)

# Unknown 처리
bogo_dis['age_group'] = bogo_dis['age_group'].astype('object').fillna('Unknown')

# 확인
bogo_dis[['age', 'age_group']].head()


,age,age_group
0,33,30s
1,33,30s
2,33,30s
5,118,70+
6,118,70+


In [40]:
# income_group 만들기
bogo_dis['income_group'] = pd.cut(
    bogo_dis['income'],
    bins=[0, 40000, 60000, 80000, 100000, np.inf],
    labels=['0-40k', '40-60k', '60-80k', '80-100k', '100k+'],
    right=False
)

# Unknown 처리
bogo_dis['income_group'] = bogo_dis['income_group'].astype('object').fillna('Unknown')

# 확인
bogo_dis[['income', 'income_group']].head()


,income,income_group
0,72000.0,60-80k
1,72000.0,60-80k
2,72000.0,60-80k
5,NaN,Unknown
6,NaN,Unknown


In [41]:
# 기본 분포 확인
print("전체 행 수:", len(bogo_dis))

print("\noffer_type 분포")
display(bogo_dis['offer_type'].value_counts())

print("\ngender 분포")
display(bogo_dis['gender'].value_counts(dropna=False))

print("\nage_group 분포")
display(bogo_dis['age_group'].value_counts(dropna=False))

print("\nincome_group 분포")
display(bogo_dis['income_group'].value_counts(dropna=False))


전체 행 수: 61042

offer_type 분포


offer_type
discount    30543
bogo        30499
Name: count, dtype: int64


gender 분포


gender
M      30562
F      21918
NaN     7841
O        721
Name: count, dtype: int64


age_group 분포


age_group
70+    18095
50s    12692
60s    10780
40s     8241
30s     5515
20s     4997
10s      722
Name: count, dtype: int64


income_group 분포


income_group
60-80k     16712
40-60k     16101
80-100k     9364
Unknown     7841
0-40k       7072
100k+       3952
Name: count, dtype: int64

In [42]:
# 전체 bogo/discount 전환율
summary = bogo_dis.groupby('offer_type').agg(
    total_offers=('person', 'count'),
    converted=('is_deduplicated', 'sum')
).reset_index()

summary['conversion_rate(%)'] = (summary['converted'] / summary['total_offers'] * 100).round(1)

display(summary)


,offer_type,total_offers,converted,conversion_rate(%)
0,bogo,30499,10600,34.8
1,discount,30543,11733,38.4


In [43]:
# 성별 전환율
gender_summary = bogo_dis.groupby('gender').agg(
    total_offers=('person', 'count'),
    converted=('is_deduplicated', 'sum')
).reset_index()

gender_summary['conversion_rate(%)'] = (gender_summary['converted'] / gender_summary['total_offers'] * 100).round(1)

display(gender_summary.sort_values('total_offers', ascending=False))


,gender,total_offers,converted,conversion_rate(%)
1,M,30562,11063,36.2
0,F,21918,10018,45.7
2,O,721,363,50.3


In [44]:
# 연령대 전환율
age_summary = bogo_dis.groupby('age_group').agg(
    total_offers=('person', 'count'),
    converted=('is_deduplicated', 'sum')
).reset_index()

age_summary['conversion_rate(%)'] = (age_summary['converted'] / age_summary['total_offers'] * 100).round(1)

display(age_summary)


,age_group,total_offers,converted,conversion_rate(%)
0,10s,722,201,27.8
1,20s,4997,1516,30.3
2,30s,5515,2018,36.6
3,40s,8241,3449,41.9
4,50s,12692,5387,42.4
5,60s,10780,4514,41.9
6,70+,18095,5248,29.0


In [45]:
# 소득 전환율
income_summary = bogo_dis.groupby('income_group').agg(
    total_offers=('person', 'count'),
    converted=('is_deduplicated', 'sum')
).reset_index()

income_summary['conversion_rate(%)'] = (income_summary['converted'] / income_summary['total_offers'] * 100).round(1)

display(income_summary)


,income_group,total_offers,converted,conversion_rate(%)
0,0-40k,7072,2016,28.5
1,100k+,3952,1785,45.2
2,40-60k,16101,5791,36.0
3,60-80k,16712,7200,43.1
4,80-100k,9364,4652,49.7
5,Unknown,7841,889,11.3


In [46]:
# Cramer's V 계산 함수
def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))


In [47]:
# 조정된 잔차 계산 함수
def adjusted_residuals(observed, expected):
    obs = observed.to_numpy(dtype=float)
    n = obs.sum()

    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)

    denom = np.sqrt(expected * (1 - row_sums / n) * (1 - col_sums / n))
    resid = np.divide(obs - expected, denom, out=np.zeros_like(obs, dtype=float), where=denom != 0)

    return pd.DataFrame(resid, index=observed.index, columns=observed.columns)


In [48]:
# 카이제곱 검정 함수
def run_chi2_test(df, group_col, target_col='is_deduplicated'):
    table = pd.crosstab(df[group_col], df[target_col])
    chi2, p_value, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    v = cramers_v(chi2, n, table.shape[0], table.shape[1])
    resid = adjusted_residuals(table, expected)

    print("=" * 80)
    print(f"{group_col} vs {target_col}")
    print("=" * 80)
    print("[교차표]")
    display(table)

    print("[카이제곱 검정]")
    print(f"chi2 = {chi2:.4f}")
    print(f"p-value = {p_value:.4e}")
    print(f"dof = {dof}")
    print(f"Cramer's V = {v:.4f}")

    print("[조정된 잔차]")
    display(resid.round(3))

    return {
        'group_col': group_col,
        'target_col': target_col,
        'chi2': chi2,
        'p_value': p_value,
        'dof': dof,
        'cramers_v': v
    }


In [49]:
# 전체 고객 특성별 검정
gender_result = run_chi2_test(bogo_dis, 'gender')
age_result = run_chi2_test(bogo_dis, 'age_group')
income_result = run_chi2_test(bogo_dis, 'income_group')


gender vs is_deduplicated
[교차표]


is_deduplicated,0,1
gender,,
F,11900,10018
M,19499,11063
O,358,363


[카이제곱 검정]
chi2 = 510.2150
p-value = 1.6152e-111
dof = 2
Cramer's V = 0.0979
[조정된 잔차]


is_deduplicated,0,1
gender,,
F,-21.251,21.251
M,22.449,-22.449
O,-5.533,5.533


age_group vs is_deduplicated
[교차표]


...

is_deduplicated,0,1
age_group,,
10s,4.909,-4.909
20s,9.570,-9.570
30s,-0.008,0.008
40s,-10.670,10.670
50s,-15.394,15.394
60s,-12.561,12.561
70+,25.250,-25.250


income_group vs is_deduplicated
[교차표]


is_deduplicated,0,1
income_group,,
0-40k,5056,2016
100k+,2167,1785
40-60k,10310,5791
60-80k,9512,7200
80-100k,4712,4652
Unknown,6952,889


[카이제곱 검정]
chi2 = 3477.4659
p-value = 0.0000e+00
dof = 5
Cramer's V = 0.2387
[조정된 잔차]


is_deduplicated,0,1
income_group,,
0-40k,15.002,-15.002
100k+,-11.580,11.580
40-60k,1.902,-1.902
60-80k,-20.460,20.460
80-100k,-28.589,28.589
Unknown,49.719,-49.719


In [50]:
# 같은 고객층 안에서 bogo vs discount 비교 결과를 저장
segment_results = []

for group_col in ['gender', 'age_group', 'income_group']:
    for group_value in sorted(bogo_dis[group_col].dropna().astype(str).unique()):
        sub = bogo_dis[bogo_dis[group_col].astype(str) == group_value].copy()
        if sub['offer_type'].nunique() < 2:
            continue

        result = run_chi2_test(sub, 'offer_type')
        result['segment'] = f'{group_col} = {group_value}'
        segment_results.append(result)


offer_type vs is_deduplicated
[교차표]


is_deduplicated,0,1
offer_type,,
bogo,5908,5067
discount,5992,4951


[카이제곱 검정]
chi2 = 1.8523
p-value = 1.7351e-01
dof = 1
Cramer's V = 0.0092
[조정된 잔차]


is_deduplicated,0,1
offer_type,,
bogo,-1.375,1.375
discount,1.375,-1.375


offer_type vs is_deduplicated
[교차표]


...

is_deduplicated,0,1
offer_type,,
bogo,-0.437,0.437
discount,0.437,-0.437


offer_type vs is_deduplicated
[교차표]


is_deduplicated,0,1
offer_type,,
bogo,3680,282
discount,3272,607


[카이제곱 검정]
chi2 = 141.0480
p-value = 1.5705e-32
dof = 1
Cramer's V = 0.1341
[조정된 잔차]


is_deduplicated,0,1
offer_type,,
bogo,11.912,-11.912
discount,-11.912,11.912


In [52]:
# 전체 요약표 만들기
summary_rows = [
    {
        'segment': 'overall',
        'group_col': gender_result['group_col'],
        'chi2': gender_result['chi2'],
        'p_value': gender_result['p_value'],
        'dof': gender_result['dof'],
        'cramers_v': gender_result['cramers_v']
    },
    {
        'segment': 'overall',
        'group_col': age_result['group_col'],
        'chi2': age_result['chi2'],
        'p_value': age_result['p_value'],
        'dof': age_result['dof'],
        'cramers_v': age_result['cramers_v']
    },
    {
        'segment': 'overall',
        'group_col': income_result['group_col'],
        'chi2': income_result['chi2'],
        'p_value': income_result['p_value'],
        'dof': income_result['dof'],
        'cramers_v': income_result['cramers_v']
    }
]

for r in segment_results:
    summary_rows.append({
        'segment': r['segment'],
        'group_col': r['group_col'],
        'chi2': r['chi2'],
        'p_value': r['p_value'],
        'dof': r['dof'],
        'cramers_v': r['cramers_v']
    })

summary_df = pd.DataFrame(summary_rows)

# 보정 p-value 계산
p = summary_df['p_value'].astype(float).to_numpy()
m = len(p)

summary_df['p_bonferroni'] = np.minimum(p * m, 1.0)

order = np.argsort(p)
bh = p[order] * m / np.arange(1, m + 1)
bh = np.minimum.accumulate(bh[::-1])[::-1]

fdr = np.empty_like(p, dtype=float)
fdr[order] = np.minimum(bh, 1.0)
summary_df['p_fdr_bh'] = fdr

display(summary_df.sort_values('p_value'))


,segment,group_col,chi2,p_value,dof,cramers_v,p_bonferroni,p_fdr_bh
2,overall,income_group,3477.465895,0.000000e+00,5,0.238681,0.000000e+00,0.000000e+00
1,overall,age_group,972.559318,7.690521e-207,6,0.126225,1.461199e-205,7.305995e-206
0,overall,gender,510.215009,1.615171e-111,2,0.097930,3.068825e-110,1.022942e-110
18,income_group = Unknown,offer_type,141.047989,1.570539e-32,1,0.134121,2.984024e-31,7.460060e-32
4,gender = M,offer_type,106.044800,7.208694e-25,1,0.058905,1.369652e-23,2.739304e-24
13,income_group = 0-40k,offer_type,63.685820,1.459312e-15,1,0.094897,2.772693e-14,4.621155e-15
15,income_group = 40-60k,offer_type,48.593725,3.148752e-12,1,0.054937,5.982628e-11,8.546612e-12
12,age_group = 70+,offer_type,35.404115,2.679202e-09,1,0.044233,5.090484e-08,6.363105e-09
14,income_group = 100k+,offer_type,15.595869,7.842573e-05,1,0.062820,1.490089e-03,1.655654e-04
9,age_group = 40s,offer_type,14.952936,1.102264e-04,1,0.042596,2.094301e-03,2.094301e-04


## bogo / discount 통계검정 인사이트

### 1. 분석 목적
- `bogo`와 `discount`의 전환이 고객 특성에 따라 달라지는지 확인
- 전체 평균이 아니라 **세그먼트별 반응 차이**를 보는 것이 핵심

### 2. 사용한 타겟
- 주 타겟: `is_deduplicated`
- 이유: last-touch 기준으로 전환 credit을 정리한 값이라, bogo/discount 성과 비교에 더 적합함

### 3. 검정 방법
- 전환 여부(`0/1`) 비교이므로 **카이제곱 검정**을 사용
- 차이의 크기는 **Cramer's V**로 확인
- 어떤 집단이 차이를 만드는지는 **조정된 잔차**로 확인

### 4. 전체 고객 특성별 결과
- 성별: `chi2 = 510.2150`, `p = 1.6152e-111`, `Cramer's V = 0.0979`
- 연령대: `chi2 = 972.5593`, `p = 7.6905e-207`, `Cramer's V = 0.1262`
- 소득수준: `chi2 = 3477.4659`, `p = 0.0000e+00`, `Cramer's V = 0.2387`

### 5. 해석
- 세 가지 특성 모두 전환과 유의한 관련이 있었음
- 영향력은 대체로 `소득수준 > 연령대 > 성별` 순
- 즉, 고객 전환은 오퍼 자체만의 문제가 아니라 **고객 특성에 따라 달라지는 반응**이라고 볼 수 있음

### 6. bogo vs discount 차이가 뚜렷했던 구간
#### 성별
- `M`: 유의함 (`p = 7.2087e-25`)
- `F`: 유의하지 않음 (`p = 0.1735`)
- `O`: 유의하지 않음 (`p = 0.9676`, 표본이 작아서 해석 주의)

#### 연령대
- `10s`: 유의함 (`p = 1.1357e-03`)
- `20s`: 유의함 (`p = 1.8278e-04`)
- `30s`: 유의함 (`p = 4.1625e-04`)
- `40s`: 유의함 (`p = 1.1023e-04`)
- `50s`: 유의하지 않음 (`p = 0.1241`)
- `60s`: 유의함 (`p = 1.4446e-04`)
- `70+`: 유의함 (`p = 2.6792e-09`)

#### 소득수준
- `0-40k`: 유의함 (`p = 1.4593e-15`)
- `40-60k`: 유의함 (`p = 3.1488e-12`)
- `60-80k`: 유의함 (`p = 7.6784e-03`)
- `80-100k`: 유의하지 않음 (`p = 0.6771`)
- `100k+`: 유의함 (`p = 7.8426e-05`)
- `Unknown`: 유의함 (`p = 1.5705e-32`, 해석은 보수적으로)

### 7. 실무적으로 읽히는 패턴
- 남성층에서는 bogo와 discount 반응 차이가 특히 뚜렷했음
- 저소득층(`0-40k`, `40-60k`)과 일부 연령대(`20s`, `30s`, `40s`, `60s`, `70+`)에서 할인형 오퍼의 반응 차이가 분명했음
- 반대로 `50s`와 `80-100k`는 두 오퍼의 반응 차이가 크지 않아 유의하지 않았음
- `O` 성별은 표본이 작아 결과가 흔들릴 수 있어 주의가 필요함

### 8. 실무 결론
- `bogo`와 `discount`는 전체 고객에게 똑같이 적용하기보다, 고객 세그먼트별로 다르게 운영하는 것이 더 합리적임
- 특히 **소득수준과 연령대**가 오퍼 반응을 가르는 중요한 기준이 될 수 있음
- 즉, 이번 검정이 보여준 핵심은 “어떤 오퍼가 좋은가”보다 **“누구에게 어떤 오퍼가 더 잘 맞는가”**이다

### 9. 한 줄 요약
- `bogo/discount` 전환은 고객 특성에 따라 달라지며, 특히 `소득수준`과 `연령대`가 중요한 차이 요인이다.


In [ ]:
#################################################################################

## 누구에게 어떤 오퍼가 더 잘 맞나

이번 검정 기준으로 보면, **discount가 더 잘 맞는 집단이 더 많았다**고 해석하는 게 가장 자연스럽다.

### discount가 상대적으로 잘 맞는 집단
- `M` 성별
- `0-40k`, `40-60k`, `60-80k`, `100k+`, `Unknown` 소득구간
- `10s`, `20s`, `30s`, `40s`, `60s`, `70+` 연령대

### bogo와 discount 차이가 뚜렷하지 않은 집단
- `F` 성별
- `50s`
- `80-100k`

### 해석
- 남성층과 저소득층에서는 discount 반응이 더 강한 편이었다.
- 일부 연령대에서도 discount 쪽이 더 잘 먹혔다.
- 반대로 여성층, 50대, 80-100k에서는 bogo와 discount 차이가 크지 않았다.
- 그래서 지금 데이터만 보면, **discount를 우선적으로 고려할 고객층이 더 넓고, bogo는 차이가 뚜렷하지 않은 구간에서 유지 후보**로 보는 게 맞다.

### 한 줄 결론
- **discount는 남성, 저소득층, 일부 연령대에 더 잘 맞고**
- **bogo는 여성, 50대, 80-100k에서는 굳이 discount보다 낫다고 말할 근거가 약하다**
